# Fine-tuned RoBERTa — Results

Evaluates document-level sentiment predictions from `bert_predictions.csv`.

- `bert_predictions.csv` contains the **full dataset** (134,068 rows)
- Ground truth column: `label` (1.0 = positive, 0.0 = negative, NaN = rating 3, excluded)
- Prediction column: `pred_sentiment` (`positive` / `negative`)
- Test split reconstructed with `test_size=0.2`, `random_state=42`, stratified by `label`

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
# Load full predictions
df = pd.read_csv("bert_predictions.csv", low_memory=False)
print(f"Total rows: {len(df)}")
print(f"\nlabel distribution:\n{df['label'].value_counts(dropna=False)}")
print(f"\npred_sentiment distribution:\n{df['pred_sentiment'].value_counts(dropna=False)}")

Total rows: 134068

label distribution:
label
1.0    105720
0.0     18523
NaN      9825
Name: count, dtype: int64

pred_sentiment distribution:
pred_sentiment
positive    111174
negative     22894
Name: count, dtype: int64


In [3]:
# Filter out rating=3 rows (label is NaN for those)
df_valid = df[df["label"].notna()].copy()
print(f"Valid rows (rating != 3): {len(df_valid)}")

# Reconstruct the same test split used during training
_, df_test = train_test_split(
    df_valid,
    test_size=0.2,
    random_state=42,
    stratify=df_valid["label"]
)
print(f"Test set size: {len(df_test)}")

Valid rows (rating != 3): 124243
Test set size: 24849


In [4]:
# Encode: positive=1, negative=0
y_true = df_test["label"].astype(int).values
y_pred = (df_test["pred_sentiment"] == "positive").astype(int).values

print("=" * 60)
print("Overall Classification Report")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=["negative", "positive"]))

acc    = accuracy_score(y_true, y_pred)
mf1    = f1_score(y_true, y_pred, average="macro")
neg_f1 = f1_score(y_true, y_pred, pos_label=0)
pos_f1 = f1_score(y_true, y_pred, pos_label=1)
print(f"Accuracy : {acc:.4f}")
print(f"Macro F1 : {mf1:.4f}")
print(f"Neg F1   : {neg_f1:.4f}")
print(f"Pos F1   : {pos_f1:.4f}")

Overall Classification Report
              precision    recall  f1-score   support

    negative       0.91      0.90      0.91      3705
    positive       0.98      0.98      0.98     21144

    accuracy                           0.97     24849
   macro avg       0.95      0.94      0.94     24849
weighted avg       0.97      0.97      0.97     24849

Accuracy : 0.9718
Macro F1 : 0.9443
Neg F1   : 0.9052
Pos F1   : 0.9834


In [5]:
# Per-topic breakdown (test set only)
df_topic = df_test[df_test["topic_label"].notna()]
topics = sorted(df_topic["topic_label"].unique())

print(f"{'Topic':<45} {'N':>6} {'Acc':>7} {'MacroF1':>9}")
print("-" * 70)

for topic in topics:
    subset = df_topic[df_topic["topic_label"] == topic]
    yt = subset["label"].astype(int).values
    yp = (subset["pred_sentiment"] == "positive").astype(int).values
    n   = len(subset)
    acc = accuracy_score(yt, yp)
    mf1 = f1_score(yt, yp, average="macro", zero_division=0)
    print(f"{topic:<45} {n:>6,} {acc:>7.3f} {mf1:>9.3f}")

Topic                                              N     Acc   MacroF1
----------------------------------------------------------------------
Accessories                                    1,779   0.967     0.914
Acoustic tone                                    870   0.990     0.876
Beginner learning                              2,579   0.985     0.937
Customer service / returns                     1,628   0.932     0.930
Electronics / controls                           415   0.954     0.908
Fret / neck setup                              2,390   0.955     0.939
Guitar size                                    1,059   0.968     0.886
Pickups                                        1,477   0.985     0.855
Playability / chords                           1,083   0.983     0.942
Setup / action                                   991   0.987     0.912
Shipping damage                                1,117   0.929     0.929
String quality                                 1,068   0.950     0.948
Tuning